[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FieldQuantum/fieldqkit/blob/main/examples/demo_vqe_h2_4q.ipynb)

In [ ]:
# 在 Google Colab 上运行请先执行本单元格安装依赖（本地已安装可跳过）
%pip install -q "fieldqkit[sim]"

In [ ]:
from pathlib import Path
import json
import math

# Geometry / chemistry config
R_UNIT = 'angstrom'
BASIS = 'sto-3g'
MULTIPLICITY = 1
CHARGE = 0

# VQE config
cfg = {
    'layers': 3,
    'shots': 4096,
    'max_iters': 50,
    'learning_rate': 0.2,
    'seed': 7,
    'gradient_method': 'autograd',
    'prefer_chips': 'Simulator',
    'shift': math.pi / 2,
}


In [ ]:
from fieldqkit import QuantumHardwareClient
from fieldqkit.algorithms import VQERunner
from fieldqkit.circuit import QuantumCircuit

def build_h2_ucc_style_symbolic_ansatz() -> QuantumCircuit:
    qc = QuantumCircuit(4)

    qc = QuantumCircuit(4)

    # -------------------------
    # Hartree–Fock reference
    # |1100>
    # -------------------------

    qc.x(0)
    qc.x(1)

    qc.pauli_evolution("theta", "Y0 Y1 Y2 X3")
    return qc

custom_ansatz_qc = build_h2_ucc_style_symbolic_ansatz()
symbolic_params = sorted(
    [k for k, v in custom_ansatz_qc.params_value.items() if isinstance(k, str) and isinstance(v, str)]
)
print('Total gates in custom H2-UCC-style ansatz:', len(custom_ansatz_qc.gates))

In [ ]:
from __future__ import annotations

import os
import json
from pathlib import Path

from fieldqkit import QuantumHardwareClient
from fieldqkit.algorithms import VQERunner
import numpy as np

R_range = np.arange(0.5, 3.01, 0.3)
energy_he = []
energy_ucc = []
for r, R in enumerate(R_range):
    json_path = Path(f'/data/h2_R{R:.1f}_angstrom_sto-3g.json')

    data = json.loads(json_path.read_text(encoding='utf-8'))

    h2_constant = float(data['constant'])
    h2_4q_terms = [(float(c), str(obs)) for c, obs in data['terms']]
    nqubits = int(data['nqubits'])
    fci_energy = float(data['fci_energy'])

    print('Loaded from:', json_path)

    client = QuantumHardwareClient()
    runner = VQERunner(
        client=client,
        shots=cfg['shots'],
        max_iters=cfg['max_iters'],
        learning_rate=cfg['learning_rate'],
        gradient_method=cfg['gradient_method'],
        seed=cfg['seed'],
    )

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_he',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='hardwareefficient',
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_he.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_ucc',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='custom',
        custom_ansatz_circuit=custom_ansatz_qc,
        init_params=[0.01]*len(symbolic_params)
        
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_ucc.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })


In [ ]:
from matplotlib import pyplot as plt

R_values = [res['R'] for res in energy_he]
estimated_energies_he = [res['estimated_total_energy'] for res in energy_he]
estimated_energies_ucc = [res['estimated_total_energy'] for res in energy_ucc]
R_range = np.arange(0.5, 3.01, 0.1)
fci_energies = []
for R in R_range:
    json_path = Path(f'/data/h2_R{R:.1f}_angstrom_sto-3g.json')
    data = json.loads(json_path.read_text(encoding='utf-8'))
    fci_energies.append(float(data['fci_energy']))
plt.figure(figsize=(10, 6))
plt.scatter(R_values, estimated_energies_he, marker='o', label='Hardware-Efficient Ansatz')
plt.scatter(R_values, estimated_energies_ucc, marker='s', label='Unitary Coupled Cluster Ansatz')
plt.plot(R_range, fci_energies, 'k-', label='FCI Energy')
plt.xlabel('Interatomic Distance R (angstrom)')
plt.ylabel('Energy (Hartree)')
plt.legend()
plt.title('H2 Molecule Energy vs Interatomic Distance')
plt.tick_params(right=True, top=True)
plt.tight_layout()
plt.show()

In [ ]:
cfg['prefer_chips'] = 'Baihua'
cfg['zne'] = True
cfg['readout_mitigation'] = True
cfg['clifford_fitting'] = True
cfg['clifford_fitting_num_samples'] = 8
cfg['max_iters'] = 10
cfg['gradient_method'] = 'parameter-shift'
energy_ucc_exp = []
R_range = np.arange(0.5, 3.01, 0.3)
for r, R in enumerate(R_range):
    json_path = Path(f'/data/h2_R{R:.1f}_angstrom_sto-3g.json')
    data = json.loads(json_path.read_text(encoding='utf-8'))

    h2_constant = float(data['constant'])
    h2_4q_terms = [(float(c), str(obs)) for c, obs in data['terms']]
    nqubits = int(data['nqubits'])
    fci_energy = float(data['fci_energy'])

    print('Loaded from:', json_path)

    client = QuantumHardwareClient()
    runner = VQERunner(
        client=client,
        shots=cfg['shots'],
        max_iters=cfg['max_iters'],
        learning_rate=cfg['learning_rate'],
        gradient_method=cfg['gradient_method'],
        seed=cfg['seed'],
        zne=cfg['zne'],
        readout_mitigation=cfg['readout_mitigation'],
        shift=cfg['shift'],
        clifford_fitting=cfg['clifford_fitting'],
        clifford_fitting_num_samples=cfg['clifford_fitting_num_samples'],
    )

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_ucc',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='custom',
        custom_ansatz_circuit=custom_ansatz_qc,
        init_params=[0.01]*len(symbolic_params)
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_ucc_exp.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })


In [ ]:
R_values = [res['R'] for res in energy_ucc_exp]
estimated_energies_ucc = [res['estimated_total_energy'] for res in energy_ucc_exp]
R_range = np.arange(0.5, 3.01, 0.1)
fci_energies = []
for R in R_range:
    json_path = Path(f'/data/h2_R{R:.1f}_angstrom_sto-3g.json')
    data = json.loads(json_path.read_text(encoding='utf-8'))
    fci_energies.append(float(data['fci_energy']))
plt.figure(figsize=(10, 6))
plt.scatter(R_values, estimated_energies_ucc, marker='s', label='Unitary Coupled Cluster Ansatz')
plt.plot(R_range, fci_energies, 'k-', label='FCI Energy')
plt.xlabel('Interatomic Distance R (angstrom)')
plt.ylabel('Energy (Hartree)')
plt.legend()
plt.title('H2 Molecule Energy vs Interatomic Distance')
plt.tick_params(right=True, top=True)
plt.tight_layout()
plt.show()